Nano Particles Data prep:

In [1]:
import pandas as pd
import numpy as np
import json
import re       
from pymatgen.core import Composition
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("data/final_data_exchange_bias.csv")

with open("data/magmom.json") as f:
    MAGMOM = json.load(f)

with open("data/ofm.json") as f:
    OFM = json.load(f)

In [3]:
# =====================================================
# Special labels found in THIS dataset
# =====================================================

SPECIAL_LABELS = {"GS"}

# =====================================================
# Composition parser
# =====================================================

def get_comp_dict(formula):

    if pd.isna(formula):
        return None

    formula = str(formula).strip()

    # no shell
    if formula == "0":
        return None

    # spin-glass label
    if formula in SPECIAL_LABELS:
        return None

    try:

        comp = Composition(formula)

        total = sum(
            comp.get_el_amt_dict().values()
        )

        return {
            el: amt/total
            for el, amt
            in comp.get_el_amt_dict().items()
        }

    except:

        return None


# =====================================================
# Weighted magnetic moment
# =====================================================

def weighted_magmom(formula):

    comp = get_comp_dict(formula)

    if comp is None:
        return 0.0

    val = 0.0

    for el, frac in comp.items():

        val += frac * MAGMOM.get(el, 0)

    return val


# =====================================================
# Verify OFM coverage
# =====================================================

missing_elements = set()

for col in ["core", "shell"]:

    for formula in df[col].dropna().unique():

        comp = get_comp_dict(formula)

        if comp is None:
            continue

        for el in comp:

            if el not in OFM:

                missing_elements.add(el)

print("Missing OFM elements:")
print(missing_elements)


# =====================================================
# Physics flags
# =====================================================

# actual shell exists
df["has_shell"] = (
    df["shell"].astype(str) != "0"
).astype(int)

# spin-glass samples
df["shell_is_gs"] = (
    df["shell"].astype(str) == "GS"
).astype(int)


# =====================================================
# Magnetic moment features
# =====================================================

df["core_magmom"] = \
df["core"].apply(weighted_magmom)

df["shell_magmom"] = \
df["shell"].apply(weighted_magmom)

df["magmom_diff"] = \
abs(
    df["core_magmom"]
    -
    df["shell_magmom"]
)

df["magmom_product"] = \
(
    df["core_magmom"]
    *
    df["shell_magmom"]
)


# =====================================================
# Verification
# =====================================================

print("\nShell statistics")

print(
    df[
        [
            "shell",
            "has_shell",
            "shell_is_gs",
            "shell_magmom"
        ]
    ]
    .sample(20)
)

print("\nCounts")

print(
    df[
        [
            "has_shell",
            "shell_is_gs"
        ]
    ]
    .sum()
)

Missing OFM elements:
set()

Shell statistics
                shell  has_shell  shell_is_gs  shell_magmom
422             CoNiO          1            0      3.176613
212                 0          0            0      0.000000
194               NiO          1            0      2.828427
439               CoO          1            0      3.350705
511             Mn2O3          1            0      4.063488
448         Mn(FeO2)2          1            0      3.861107
526             Mn2O3          1            0      4.063488
825                 0          0            0      0.000000
85              Fe3O4          1            0      3.715807
353                Mo          1            0      6.928203
397               NiO          1            0      2.828427
40                  0          0            0      0.000000
141               CoO          1            0      3.350705
365                Co          1            0      3.872983
395                 0          0            0      0.0

In [4]:
def magnetic_fraction(formula):

    comp = get_comp_dict(formula)

    if comp is None:
        return 0.0

    magnetic_elements = {
        "Sc","Ti","V","Cr","Mn",
        "Fe","Co","Ni","Cu","Zn",
        "Y","Zr","Nb","Mo","Tc",
        "Ru","Rh","Pd",
        "Ce","Pr","Nd","Sm",
        "Eu","Gd","Tb","Dy",
        "Ho","Er","Tm","Yb"
    }

    frac = 0

    for el,w in comp.items():

        if el in magnetic_elements:
            frac += w

    return frac

In [5]:
df["core_mag_fraction"] = \
df["core"].apply(magnetic_fraction)

df["shell_mag_fraction"] = \
df["shell"].apply(magnetic_fraction)

df["core_mag_density"] = \
df["core_magmom"] * df["core_mag_fraction"]

df["shell_mag_density"] = \
df["shell_magmom"] * df["shell_mag_fraction"]

In [6]:
OFM_DIM = 32

def weighted_ofm(formula):

    vec = np.zeros(OFM_DIM)

    comp = get_comp_dict(formula)

    if comp is None:
        return vec

    for el, frac in comp.items():

        vec += frac * np.array(OFM[el])

    return vec

In [7]:
print(len(weighted_ofm("Fe")))

print(weighted_ofm("Fe")[:10])

print(weighted_ofm("Fe2CoO4")[:10])

32
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
[0.         1.         0.         0.         0.         0.57142857
 0.         0.         0.         0.        ]


In [8]:
missing_elements = set()

for col in ["core","shell"]:

    for formula in df[col].dropna().unique():

        comp = get_comp_dict(formula)

        if comp is None:
            continue

        for el in comp:

            if el not in OFM:

                missing_elements.add(el)

print(missing_elements)

set()


In [9]:
df["has_shell"] = (
    df["shell"].astype(str) != "0"
).astype(int)

df["shell_is_gs"] = (
    df["shell"].astype(str) == "GS"
).astype(int)

In [10]:
print(df[["has_shell","shell_is_gs"]].sum())

has_shell      767
shell_is_gs      8
dtype: int64


In [11]:
print("Unique core values:")
print(sorted(df["core"].astype(str).unique())[:50])

print("\nUnique shell values:")
print(sorted(df["shell"].astype(str).unique())[:50])

Unique core values:
['Ag', 'Al', 'AlNi3', 'Au', 'BaFe12O19', 'Ca', 'CaMnO3', 'CaNdMn2O6', 'CaSm', 'CaSmMn2O6', 'Ce', 'Co', 'Co2NiO4', 'Co3O4', 'CoNiO', 'CoO', 'CoP', 'Cr2CoO4', 'Cr2CuO4', 'Cr2O3', 'Cr2Te3', 'CrAgO2', 'CrN', 'CrO2', 'CrO3', 'CuO', 'DyTiCrO5', 'Er3Fe2(NiO4)25', 'Er3Fe7(NiO4)50', 'Er9Fe(NiO4)50', 'Fe', 'Fe11C5', 'Fe11Co49O80', 'Fe13(CoO10)2', 'Fe17O60', 'Fe19O60', 'Fe23Co77', 'Fe23Co7O40', 'Fe2CoO4', 'Fe2NiO4', 'Fe2O3', 'Fe3Co7', 'Fe3Co9O4', 'Fe3O4', 'Fe43Co7O50', 'Fe4Bi2O9', 'Fe5C2', 'Fe6Co19', 'Fe7Co3', 'Fe7O30']

Unique shell values:
['0', 'Ag', 'Au', 'BaTiO3', 'CO', 'Co', 'Co2NiO4', 'Co3O4', 'CoNi', 'CoNiO', 'CoO', 'Cr', 'Cr(FeO3)2', 'Cr2O3', 'CrO2', 'CuO', 'Fe', 'Fe101Co99O400', 'Fe200Te79', 'Fe250Te63', 'Fe2Co', 'Fe2CoO4', 'Fe2CuO4', 'Fe2NiO4', 'Fe2O3', 'Fe37(Co19O50)2', 'Fe3Co9O16', 'Fe3O4', 'Fe41Co159O400', 'Fe4Bi2O9', 'Fe4CoNiO8', 'Fe6CoO10', 'Fe7Co13O40', 'Fe8Te9', 'FeBiO3', 'FeCo', 'FeO', 'FeTe', 'FeTe2', 'GS', 'La2MnCoO6', 'La2MnO6', 'LaFeO3', 'LaMn0.5Fe0.5O3'

In [12]:
MAG_LABELS = {
    "GS","SG","SD",
    "FM","AFM","FiM",
    "PM"
}

for label in MAG_LABELS:

    core_count = (df["core"] == label).sum()
    shell_count = (df["shell"] == label).sum()

    print(
        f"{label:4s}  core={core_count:4d}  shell={shell_count:4d}"
    )

PM    core=   0  shell=   0
FM    core=   0  shell=   0
GS    core=   0  shell=   8
FiM   core=   0  shell=   0
SG    core=   0  shell=   0
SD    core=   0  shell=   0
AFM   core=   0  shell=   0


In [13]:
valid = 0
invalid = 0

for col in ["core","shell"]:

    for x in df[col]:

        comp = get_comp_dict(x)

        if comp is None:
            invalid += 1
        else:
            valid += 1

print("Valid compositions :", valid)
print("Non-compositions   :", invalid)

Valid compositions : 1738
Non-compositions   : 220


In [14]:
non_comps = set()

for col in ["core","shell"]:

    for x in df[col].unique():

        if get_comp_dict(x) is None:

            non_comps.add(str(x))

print(sorted(non_comps))

['0', 'GS']


In [21]:
df["core_magmom"] = \
df["core"].apply(weighted_magmom)

df["shell_magmom"] = \
df["shell"].apply(weighted_magmom)

print(
    df[
        ["core",
         "core_magmom",
         "shell",
         "shell_magmom"]
    ].head(20)
)

           core  core_magmom    shell  shell_magmom
0           ZnO     1.414214  Fe2CoO4      3.569236
1           CoO     3.350705  Fe2CoO4      3.569236
2   MnZn(FeO2)4     3.438530        0      0.000000
3   MnZn(FeO2)4     3.438530        0      0.000000
4   MnZn(FeO2)4     3.438530        0      0.000000
5   MnZn(FeO2)4     3.438530        0      0.000000
6   MnZn(FeO2)4     3.438530        0      0.000000
7        LaFeO3     3.023262        0      0.000000
8        LaFeO3     3.023262        0      0.000000
9        LaFeO3     3.023262        0      0.000000
10       LaFeO3     3.023262        0      0.000000
11       LaFeO3     3.023262        0      0.000000
12       LaFeO3     3.023262        0      0.000000
13       LaFeO3     3.023262        0      0.000000
14           Ni     2.828427      NiO      2.828427
15           Ni     2.828427      NiO      2.828427
16           Ni     2.828427      NiO      2.828427
17           Ni     2.828427      NiO      2.828427
18          

In [23]:
print(df[df["shell"]=="GS"])

     area/volume  h_range_max_koe      Tc  temperature_k  sat_em_g    coer_oe  \
887          2.0             90.0   678.2          150.0    0.4150   7381.608   
888          2.0             90.0  1119.4          120.0    0.7850    708.850   
889          2.0             90.0  1119.4           90.0    1.1350   6371.850   
890          2.0             90.0  1119.4           90.0    1.1500   6462.265   
891          2.0             90.0  1119.4           10.0    0.5000   7358.490   
892          2.0             90.0   446.4            5.0    3.7550  15645.500   
893          2.0             90.0   361.8           50.0    3.8165  11452.750   
894          2.0             90.0   666.8          100.0    3.3370   5698.925   

     mr (emu/g)  exc_bias_oe  exc_dir  ver_shift_emu_g  ...  has_shell  \
887      5.2547     1171.432      0.0           0.0000  ...          1   
888      0.2205        1.000      0.0           0.0035  ...          1   
889      0.6330      660.750      0.0           

In [24]:
OFM_DIM = 32

def weighted_ofm(formula):

    vec = np.zeros(OFM_DIM)

    comp = get_comp_dict(formula)

    if comp is None:
        return vec

    for el, frac in comp.items():

        vec += frac*np.array(OFM[el])

    return vec

In [25]:
core_ofm = [
    weighted_ofm(x)
    for x in df["core"]
]

shell_ofm = [
    weighted_ofm(x)
    for x in df["shell"]
]

In [26]:
df["ofm_distance"] = [
    np.linalg.norm(c-s)
    for c,s in zip(core_ofm,shell_ofm)
]

df["ofm_dot"] = [
    np.dot(c,s)
    for c,s in zip(core_ofm,shell_ofm)
]

df["ofm_cosine"] = [

    cosine_similarity(
        c.reshape(1,-1),
        s.reshape(1,-1)
    )[0,0]

    if np.linalg.norm(c)>0
    and np.linalg.norm(s)>0

    else 0

    for c,s in zip(core_ofm,shell_ofm)
]

In [27]:
print(
    df[
        [
            "core",
            "shell",
            "ofm_distance",
            "ofm_dot",
            "ofm_cosine"
        ]
    ].sample(20)
)

        core     shell  ofm_distance   ofm_dot  ofm_cosine
509    Fe3O4     Fe2O3      0.040406  1.514286    0.999466
375   LaFeO3         0      1.200000  0.000000    0.000000
880      MnO     Mn3O4      0.101015  1.500000    0.996616
749      NiO   Fe2CoO4      0.597614  1.285714    0.878310
551      NiO         0      1.224745  0.000000    0.000000
177      NiO       NiO      0.000000  1.500000    1.000000
975   FeBiO3         0      1.232883  0.000000    0.000000
794     FeCo     Fe2O3      0.787401  1.200000    0.794719
54    FeBiO3  Fe4Bi2O9      0.133333  1.493333    0.994152
967   Cr2Te3    Fe8Te9      0.843456  1.235294    0.782081
283    Fe3O4       NiO      0.662401  1.285714    0.854242
817      CoO      FeCo      0.707107  1.250000    0.833333
21   Fe2CoO4    FeBiO3      0.385450  1.400000    0.950069
712    Fe3O4       CoO      0.662401  1.285714    0.854242
956      FeO     ZnFeO      0.408248  1.333333    0.942809
440       Co       CoO      0.707107  1.500000    0.8660

In [28]:
print(
    df[
        [
            "ofm_distance",
            "ofm_dot",
            "ofm_cosine"
        ]
    ].describe()
)

       ofm_distance     ofm_dot  ofm_cosine
count    979.000000  979.000000  979.000000
mean       0.750193    1.001818    0.645621
std        0.460096    0.600503    0.389688
min        0.000000    0.000000    0.000000
25%        0.414404    0.550000    0.384789
50%        0.707107    1.285714    0.854242
75%        1.200000    1.493333    0.941214
max        2.123724    1.520000    1.000000


In [29]:
print(df[df["shell"]=="CO"])

     area/volume  h_range_max_koe      Tc  temperature_k  sat_em_g   coer_oe  \
694     0.020000             70.0  1081.0            5.0    6.1098  2552.300   
695     0.176471             70.0   891.0            5.0    1.4325  1756.100   
696     0.003000             70.0   938.4            5.0    4.0565  3112.500   
697     0.176471             70.0   432.8            5.0    8.8963  1270.137   
698     0.176471             70.0   889.4            5.0    5.3413  1374.400   
699     0.176471             70.0   891.0            5.0    1.5085  1402.836   
700     0.020000             70.0  1026.4            5.0   12.4824  1727.219   
701     0.020000             70.0  1191.0            5.0   12.4824  1739.615   
702     0.020000             70.0  1081.0            5.0   12.4824  1673.497   
703     0.020000             70.0  1191.0            5.0   10.1340  1674.500   
704     0.003000             70.0   648.8            5.0    4.0565  3099.490   
705     0.003000             70.0   648.

In [30]:
corr = (
    df.corr(numeric_only=True)
    ["exc_bias_oe"]
    .sort_values(
        key=abs,
        ascending=False
    )
)

print(corr.head(40))

exc_bias_oe                                   1.000000
coer_oe                                       0.261809
h_range_max_koe                               0.241905
temperature_k                                -0.210092
fc_field_t                                    0.168948
exc_dir                                      -0.153386
core_pauling_electronegativity_difference     0.141642
core_vec_difference                           0.136400
core_sum_vec                                 -0.134179
core_sum_work_function                       -0.125145
ver_s_dir                                     0.124141
ofm_distance                                  0.106755
magmom_diff                                   0.101026
shell_mag_fraction                           -0.100052
mr (emu/g)                                   -0.099879
core_work_function_difference                 0.098561
ofm_dot                                      -0.098342
ofm_cosine                                   -0.094157
shell_sum_

In [31]:
print(magnetic_fraction("NiO"))
print(magnetic_fraction("CoO"))
print(magnetic_fraction("Fe3O4"))
print(magnetic_fraction("MnO"))

0.5
0.5
0.42857142857142855
0.5


In [32]:
print("Fe")
print(np.where(np.array(OFM["Fe"]) > 0)[0])

print("Co")
print(np.where(np.array(OFM["Co"]) > 0)[0])

print("Ni")
print(np.where(np.array(OFM["Ni"]) > 0)[0])

print("La")
print(np.where(np.array(OFM["La"]) > 0)[0])

Fe
[ 1 13]
Co
[ 1 14]
Ni
[ 1 15]
La
[1 8]


In [33]:
def orbital_summary(ofm):

    s_occ = np.sum(ofm[0:2])

    p_occ = np.sum(ofm[2:8])

    d_occ = np.sum(ofm[8:18])

    f_occ = np.sum(ofm[18:32])

    return s_occ,p_occ,d_occ,f_occ

In [34]:
core_s=[]
core_p=[]
core_d=[]
core_f=[]

shell_s=[]
shell_p=[]
shell_d=[]
shell_f=[]

for c,s in zip(core_ofm,shell_ofm):

    cs,cp,cd,cf = orbital_summary(c)
    ss,sp,sd,sf = orbital_summary(s)

    core_s.append(cs)
    core_p.append(cp)
    core_d.append(cd)
    core_f.append(cf)

    shell_s.append(ss)
    shell_p.append(sp)
    shell_d.append(sd)
    shell_f.append(sf)

In [35]:
df["core_d_occ"] = core_d
df["shell_d_occ"] = shell_d

df["core_f_occ"] = core_f
df["shell_f_occ"] = shell_f

In [38]:
df["d_occ_diff"] = abs(
    df["core_d_occ"]
    -
    df["shell_d_occ"]
)

df["f_occ_diff"] = abs(
    df["core_f_occ"]
    -
    df["shell_f_occ"]
)

In [39]:
print(
    df[
        [
            "core",
            "shell",
            "core_d_occ",
            "shell_d_occ",
            "d_occ_diff",
            "core_f_occ",
            "shell_f_occ",
            "f_occ_diff"
        ]
    ].sample(20)
)

          core     shell  core_d_occ  shell_d_occ  d_occ_diff  core_f_occ  \
40        MnO2         0    0.333333     0.000000    0.333333         0.0   
321      Fe3O4       MgO    0.428571     0.000000    0.428571         0.0   
53      FeBiO3  Fe4Bi2O9    0.400000     0.400000    0.000000         0.2   
293         Co       CoO    1.000000     0.500000    0.500000         0.0   
824      Fe2O3         0    0.400000     0.000000    0.400000         0.0   
857         Co     Cr2O3    1.000000     0.400000    0.600000         0.0   
569        CoO     Fe3O4    0.500000     0.428571    0.071429         0.0   
346  Mn(FeO2)2     Fe2O3    0.428571     0.400000    0.028571         0.0   
840     MnGeTe       MnO    1.000000     0.500000    0.500000         0.0   
714        FeO     Fe3O4    0.500000     0.428571    0.071429         0.0   
619         Fe     Fe3O4    1.000000     0.428571    0.571429         0.0   
173    Fe2CoO4       CoO    0.428571     0.500000    0.071429         0.0   

In [40]:
corr = (
    df.corr(numeric_only=True)
      ["exc_bias_oe"]
      .sort_values(
          key=abs,
          ascending=False
      )
)

print(corr.head(50))

exc_bias_oe                                   1.000000
coer_oe                                       0.261809
h_range_max_koe                               0.241905
temperature_k                                -0.210092
fc_field_t                                    0.168948
exc_dir                                      -0.153386
core_pauling_electronegativity_difference     0.141642
core_vec_difference                           0.136400
core_sum_vec                                 -0.134179
core_sum_work_function                       -0.125145
ver_s_dir                                     0.124141
core_f_occ                                    0.118117
ofm_distance                                  0.106755
magmom_diff                                   0.101026
shell_mag_fraction                           -0.100052
mr (emu/g)                                   -0.099879
core_work_function_difference                 0.098561
ofm_dot                                      -0.098342
ofm_cosine

In [41]:
df.to_csv(
    "exchange_bias_engineered_full.csv",
    index=False
)

print(df.shape)
print("Master dataset saved.")

(979, 60)
Master dataset saved.


In [42]:
ml_df = df.copy()

drop_cols = [
    "core",
    "shell",
    "formula"
]

ml_df = ml_df.drop(
    columns=drop_cols,
    errors="ignore"
)

print(ml_df.shape)

(979, 57)


In [43]:
missing = ml_df.isna().sum()

print(
    missing[missing > 0]
)

Series([], dtype: int64)


In [44]:
target = "exc_bias_oe"

print(target in ml_df.columns)

True


In [45]:
X = ml_df.drop(columns=["exc_bias_oe"])
y = ml_df["exc_bias_oe"]

print("Features:", X.shape[1])
print("Samples :", X.shape[0])

Features: 56
Samples : 979


In [46]:
constant_cols = [
    c for c in X.columns
    if X[c].nunique() <= 1
]

print("Constant columns:")
print(constant_cols)
print("Count:", len(constant_cols))

Constant columns:
[]
Count: 0


In [47]:
dup_cols = []

cols = X.columns

for i in range(len(cols)):
    for j in range(i+1, len(cols)):

        if X[cols[i]].equals(X[cols[j]]):

            dup_cols.append(
                (cols[i], cols[j])
            )

print(dup_cols)

[]


In [48]:
corr = X.corr().abs()

upper = corr.where(
    np.triu(
        np.ones(corr.shape),
        k=1
    ).astype(bool)
)

high_corr = [
    column
    for column in upper.columns
    if any(upper[column] > 0.95)
]

print(high_corr)

['shell_beta', 'ofm_cosine']


In [49]:
X["exc_bias_oe"] = y

X.to_csv(
    "exchange_bias_final_training_dataset.csv",
    index=False
)

print(X.shape)

(979, 57)


In [50]:
baseline_remove = [

    # magmom features
    "core_magmom",
    "shell_magmom",
    "magmom_diff",
    "magmom_product",

    # magnetic fraction features
    "core_mag_fraction",
    "shell_mag_fraction",

    # magnetic density features
    "core_mag_density",
    "shell_mag_density",

    # shell flags
    "has_shell",
    "shell_is_gs",

    # OFM similarity features
    "ofm_distance",
    "ofm_dot",
    "ofm_cosine",

    # orbital descriptors
    "core_d_occ",
    "shell_d_occ",
    "core_f_occ",
    "shell_f_occ",

    "d_occ_diff",
    "f_occ_diff"
]

baseline_df = ml_df.drop(
    columns=baseline_remove,
    errors="ignore"
)

print("Baseline shape:")
print(baseline_df.shape)

baseline_df.to_csv(
    "exchange_bias_baseline.csv",
    index=False
)

Baseline shape:
(979, 38)


In [51]:
enhanced_df = ml_df.copy()

print("Enhanced shape:")
print(enhanced_df.shape)

enhanced_df.to_csv(
    "exchange_bias_enhanced.csv",
    index=False
)

Enhanced shape:
(979, 57)


In [52]:
print("\nBaseline columns:")
print(len(baseline_df.columns))

print("\nEnhanced columns:")
print(len(enhanced_df.columns))

print(
    "\nExtra features added:",
    len(enhanced_df.columns)
    -
    len(baseline_df.columns)
)


Baseline columns:
38

Enhanced columns:
57

Extra features added: 19


In [53]:
baseline_features = set(baseline_df.columns)
enhanced_features = set(enhanced_df.columns)

extra_features = sorted(
    enhanced_features - baseline_features
)

print("Number of extra features:",
      len(extra_features))

for f in extra_features:
    print(f)

Number of extra features: 19
core_d_occ
core_f_occ
core_mag_density
core_mag_fraction
core_magmom
d_occ_diff
f_occ_diff
has_shell
magmom_diff
magmom_product
ofm_cosine
ofm_distance
ofm_dot
shell_d_occ
shell_f_occ
shell_is_gs
shell_mag_density
shell_mag_fraction
shell_magmom


In [12]:
def magmom_stats(formula):

    comp = get_comp_dict(formula)

    if comp is None:
        return [np.nan]*4

    vals = []

    for el, frac in comp.items():

        vals.extend(
            [MAGMOM.get(el,0)] * int(max(1,frac*100))
        )

    vals = np.array(vals)

    return [
        vals.mean(),
        vals.std(),
        vals.max(),
        vals.min()
    ]

In [13]:
def weighted_ofm(formula):

    comp = get_comp_dict(formula)

    vec = np.zeros(32)

    if comp is None:
        return vec

    for el, frac in comp.items():

        vec += frac*np.array(OFM[el])

    return vec

In [14]:
def orbital_summary(ofm):

    s_occ = np.sum(ofm[0:8])

    p_occ = np.sum(ofm[8:16])

    d_occ = np.sum(ofm[16:24])

    f_occ = np.sum(ofm[24:32])

    return s_occ,p_occ,d_occ,f_occ

In [15]:
core_ofm = []
shell_ofm = []

for _,row in df.iterrows():

    core_ofm.append(
        weighted_ofm(row["core"])
    )

    shell_ofm.append(
        weighted_ofm(row["shell"])
    )

In [16]:
df["core_magmom"] = \
df["core"].apply(weighted_magmom)

df["shell_magmom"] = \
df["shell"].apply(weighted_magmom)

df["magmom_diff"] = \
abs(
    df["core_magmom"] -
    df["shell_magmom"]
)

df["magmom_ratio"] = \
(
    df["core_magmom"] + 1e-6
) / (
    df["shell_magmom"] + 1e-6
)

In [17]:
ofm_distance = []
ofm_cosine = []

for c,s in zip(core_ofm,shell_ofm):

    dist = np.linalg.norm(c-s)

    sim = cosine_similarity(
        c.reshape(1,-1),
        s.reshape(1,-1)
    )[0][0]

    ofm_distance.append(dist)

    ofm_cosine.append(sim)

df["ofm_distance"] = ofm_distance

df["ofm_cosine"] = ofm_cosine

In [18]:
core_d = []
shell_d = []

core_p = []
shell_p = []

core_f = []
shell_f = []

for c,s in zip(core_ofm,shell_ofm):

    cs,cp,cd,cf = orbital_summary(c)

    ss,sp,sd,sf = orbital_summary(s)

    core_d.append(cd)
    shell_d.append(sd)

    core_p.append(cp)
    shell_p.append(sp)

    core_f.append(cf)
    shell_f.append(sf)

df["core_d_occ"] = core_d
df["shell_d_occ"] = shell_d

df["core_p_occ"] = core_p
df["shell_p_occ"] = shell_p

df["core_f_occ"] = core_f
df["shell_f_occ"] = shell_f

df["d_occ_diff"] = \
abs(df["core_d_occ"]-df["shell_d_occ"])

df["p_occ_diff"] = \
abs(df["core_p_occ"]-df["shell_p_occ"])

df["f_occ_diff"] = \
abs(df["core_f_occ"]-df["shell_f_occ"])

In [19]:
interaction = []

for c,s in zip(core_ofm,shell_ofm):

    interaction.append(
        np.dot(c,s)
    )

df["ofm_interaction"] = interaction

In [20]:
df["magnetic_interaction"] = (
    df["core_magmom"] *
    df["shell_magmom"]
)

In [21]:
df.to_csv(
    "exchange_bias_engineered.csv",
    index=False
)

In [22]:
drop_cols = [
    "core",
    "shell",
    "exc_bias_oe"
]

X = df.drop(
    columns=drop_cols
)

y = df["exc_bias_oe"]

In [23]:
print(df.shape)

print(df.isna().sum().sort_values(ascending=False).head(30))

(979, 58)
magnetic_interaction                          220
magmom_diff                                   220
shell                                         220
shell_magmom                                  220
magmom_ratio                                  220
d_occ_diff                                      0
shell_f_occ                                     0
shell_sum_work_function                         0
core_pauling_electronegativity_difference       0
core_vec_difference                             0
core_work_function_difference                   0
shell_pauling_electronegativity_difference      0
shell_vec_difference                            0
shell_work_function_difference                  0
core                                            0
ofm_interaction                                 0
formula                                         0
core_magmom                                     0
f_occ_diff                                      0
p_occ_diff                              

In [9]:
for _, row in df.iterrows():

    try:
        weighted_ofm(row["shell"])

    except Exception as e:

        print("Formula:", row["shell"])
        print("Error:", e)
        break

Formula: GS
Error: 'G'


In [10]:
from pymatgen.core import Composition

bad_formulas = set()

for col in ["core", "shell"]:

    for formula in df[col].dropna().unique():

        try:
            comp = Composition(str(formula))

            for el in comp.get_el_amt_dict():

                if el not in OFM:
                    bad_formulas.add(formula)

        except:
            bad_formulas.add(formula)

print("Bad formulas:")
for x in sorted(bad_formulas):
    print(x)

Bad formulas:
0
GS
